# Generate DNA sequence from Amino Acid Sequence for TwistBio

In [1]:
%pip install -q dnachisel biopython requests pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import itertools
import pandas as pd
from Bio import Restriction
from dnachisel import (
    DnaOptimizationProblem,
    reverse_translate,
    CodonOptimize,
    AvoidPattern,
    EnforceTranslation,
    EnzymeSitePattern
)

# "combinatorial" = Multiply (Loop1_VarA + Loop2_VarA, Loop1_VarA + Loop2_VarB...)
# "positional"    = Add      (Loop1_VarA + Loop2_WT, Loop1_WT + Loop2_VarA...)
GENERATION_MODE = "positional"

# Your Input Data
CANONICAL_SEQ = "CSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKMYRGFTKMPHVQYIHTEASESLCGLKLEVNKYQYLLTGRVYDGKMYTGLCNFVERWDQLTLSQRKGLNYRYHLGCNCKIKSCYYLPCFVTSKNECLWTDMLSNFGYPGYQSKHYACIRQKGGYCSWYRGWAPPDKSIINATDP" # TIMP3
GENE_NAME = "TIMP3YeastGene_01"

#Twist API Credentials (You must request these from Twist)
API_BASE_URL = "https://twist-api.twistbioscience.com/v1"
API_TOKEN = "YOUR_TWIST_API_TOKEN"
USER_EMAIL = "your_email@example.com"

# 3. Optimization Settings
HOST_ORGANISM = "s_cerevisiae" 
PROJECT_NAME = "Yeast_Loop_Library_2025"

OUTPUT_FOLDER = "../Local/Twist_Order_Dec2025"

# Loops are positioned 2 AA down from normal because of the removal of "CT" from the canonical seq
DESIGN_SPECS = {
    'AB_LOOP': {
        'range': (28, 34), 
        'variants': ["tlpdgske", "knpdgtlt", "kgpyge", "patptstrgaggee", "eversghkvke", "tdtfptanwtgev", "dgptge"] 
    },
    'C_LOOP': {
        'range': (60, 66), 
        'variants': ["asgpitvngetiw", "ltqeelpdpnavspc", "sveslc", "asveavetgfs", "anpeyc", "ggnygsck"]
    }
}

# List of "Known Endonucleases" to avoid (Common cloning sites + Golden Gate)
RESTRICTION_SITES_TO_AVOID = [
    "BsrGI_site", "BamHI_site",
    "BsaI_site" # Added good practice for Golden Gate compatibility
]

In [11]:
def setup_output_folder(folder_path):
    """Creates the output directory if it doesn't exist."""
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Created output directory: {folder_path}")
    else:
        print(f"Using existing output directory: {folder_path}")

def get_enzyme_name(site_string):
    """Strips '_site' suffix to get standard enzyme name."""
    return site_string.replace("_site", "")

def get_enzyme_constraint(site_name):
    """
    Converts user-friendly names (e.g., 'BsrGI_site') 
    into DNA Chisel constraints.
    """
    clean_name = site_name.replace("_site", "")
    # EnzymeSitePattern looks up the cut sequence for the enzyme name
    return AvoidPattern(EnzymeSitePattern(clean_name))

def get_enzyme_sequence(site_string):
    """
    Looks up the actual cut sequence using Biopython.
    Used for the Validation Report.
    """
    name = get_enzyme_name(site_string)
    try:
        # Dynamic lookup from Biopython's Restriction library
        enzyme_obj = getattr(Restriction, name)
        return enzyme_obj.site
    except AttributeError:
        print(f"CRITICAL WARNING: Enzyme '{name}' not found in Biopython database.")
        return "NNNNNN" # Fail-safe

In [12]:
def generate_combinations(canonical, specs, mode="combinatorial"):
    """
    Generates all combinatorial variants.
    """
    loop_names = sorted(specs.keys())
    combined_designs = []

    # Ensure canonical is upper case
    canonical = canonical.upper()
    
    # ---------------------------------------
    # MODE A: COMBINATORIAL (Cartesian Product)
    # ---------------------------------------
    if mode == "combinatorial":
        print(f"   > Mode: Combinatorial (Multiplying variants)")
        lists_of_variants = [specs[name]['variants'] for name in loop_names]
        
        for combination in itertools.product(*lists_of_variants):
            current_seq = list(canonical)
            variant_tags = []
            
            # Sort reverse to preserve indices
            sorted_loops = sorted(loop_names, key=lambda x: specs[x]['range'][0], reverse=True)
            
            for loop_name in sorted_loops:
                start, end = specs[loop_name]['range']
                # Find the variant for this loop
                val = next(v for v in combination if v in specs[loop_name]['variants'])
                val = val.upper()
                
                current_seq[start:end] = list(val)
                variant_tags.append(f"{loop_name}-{val}")
                
            final_aa_seq = "".join(current_seq)
            # Name: Var_LOOP1-VAL_LOOP2-VAL
            variant_name = f"Var_{'_'.join(variant_tags[::-1])}"
            
            combined_designs.append({ "name": variant_name, "aa_sequence": final_aa_seq })

    # ---------------------------------------
    # MODE B: POSITIONAL (One Loop at a Time)
    # ---------------------------------------
    elif mode == "positional":
        print(f"   > Mode: Positional (Adding variants, others stay WT)")
        
        for loop_name in loop_names:
            variants = specs[loop_name]['variants']
            start, end = specs[loop_name]['range']
            
            for val in variants:
                val = val.upper()
                current_seq = list(canonical)
                
                # Apply ONLY the current loop variant
                # All other loops remain as they are in CANONICAL_SEQ (Wild Type)
                current_seq[start:end] = list(val)
                
                final_aa_seq = "".join(current_seq)
                
                # Name: Var_LOOP1-VAL (Implies others are WT)
                variant_name = f"Var_{loop_name}-{val}"
                
                combined_designs.append({ "name": variant_name, "aa_sequence": final_aa_seq })
                
    else:
        raise ValueError("GENERATION_MODE must be 'combinatorial' or 'positional'")

    return combined_designs

def optimize_variant(design_entry):
    print(f"  > Optimizing codons for {HOST_ORGANISM}...") # Optional noise
    print(f"Optimizing {design_entry['name']}...")
    
    naive_dna = reverse_translate(design_entry['aa_sequence'])
    
    constraints = [EnforceTranslation()]
    
    # Add named restriction sites dynamically
    for site in RESTRICTION_SITES_TO_AVOID:
        try:
            constraints.append(get_enzyme_constraint(site))
        except Exception as e:
            print(f"Warning: Could not process restriction site {site}. {e}")

    # Standard synthesis constraints
    constraints.extend([
        AvoidPattern("9xA"), AvoidPattern("9xT"),
        AvoidPattern("9xG"), AvoidPattern("9xC")
    ])
    
    problem = DnaOptimizationProblem(
        sequence=naive_dna,
        constraints=constraints,
        objectives=[CodonOptimize(species=HOST_ORGANISM)],
        logger=None
    )
    
    problem.resolve_constraints()
    problem.optimize()
    
    return problem.sequence

def validate_and_report(variants_data, folder):
    """
    Look up sequences for the sites in RESTRICTION_SITES_TO_AVOID
    and verify they are absent from the optimized DNA.
    """
    report_path = os.path.join(folder, "validation_report.txt")
    print(f"\n--- Running Validation Checks ---")
    
    with open(report_path, "w") as f:
        f.write("VALIDATION REPORT\n")
        f.write("=================\n")
        
        # 1. Resolve patterns once
        patterns = {}
        for site in RESTRICTION_SITES_TO_AVOID:
            seq = get_enzyme_sequence(site)
            patterns[get_enzyme_name(site)] = seq
            f.write(f"Avoid Target: {get_enzyme_name(site)} = {seq}\n")
        
        f.write("\n")
        
        passes = 0
        fails = 0
        
        for v in variants_data:
            dna = v['dna']
            issues = []
            
            # Check for enzyme sites
            for enzyme, pattern in patterns.items():
                if pattern in dna:
                    issues.append(f"FAIL: Found {enzyme} ({pattern})")
            
            # Check for homopolymers
            for base in ['A', 'T', 'G', 'C']:
                if base * 10 in dna:
                    issues.append(f"WARN: Homopolymer run {base}x10")

            status = "PASS" if not issues else "FAIL"
            if status == "PASS": passes += 1
            else: fails += 1
            
            f.write(f"[{status}] {v['name']}\n")
            if issues:
                for issue in issues:
                    f.write(f"       !!! {issue}\n")
            f.write("-" * 50 + "\n")
            
        summary = f"SUMMARY: {passes} Passed, {fails} Failed."
        f.write(f"\n{summary}")
        print(summary)

def save_output_files(data_list, folder):
    # Paths
    fasta_path = os.path.join(folder, "twist_library.fasta")
    csv_path = os.path.join(folder, "twist_library.csv")

    # Save FASTA
    with open(fasta_path, "w") as f:
        for seq in data_list:
            f.write(f">{seq['name']}\n")
            f.write(f"{seq['dna']}\n")
    
    # Save CSV
    df = pd.DataFrame(data_list)
    df = df[["name", "dna", "aa_seq"]] 
    df.columns = ["Construct Name", "DNA Sequence", "Amino Acid Sequence"]
    df.to_csv(csv_path, index=False)
    
    print(f"Saved FASTA: {fasta_path}")
    print(f"Saved CSV:   {csv_path}")

def upload_draft_to_twist(sequences):
    """
    Uploads sequences as a 'Draft' project (Design phase).
    Does NOT execute an order.
    """
    print(f"\n--- Uploading {len(sequences)} sequences to Twist (Draft) ---")
    
    url = f"{API_BASE_URL}/constructs" 
    headers = {
        "Authorization": f"Bearer {API_TOKEN}",
        "Content-Type": "application/json"
    }
    
    # Twist API accepts batch creation
    # We set properties to ensure it remains a draft/design object
    payload_items = []
    for s in sequences:
        payload_items.append({
            "name": s['name'],
            "sequences": [s['dna']],
            "type": "CLONED_GENE", # Change to NON_CLONED_GENE if just fragments
            # "vector_mes_uid": "GET_THIS_FROM_PORTAL", 
            # "insertion_point_mes_uid": "GET_THIS_FROM_PORTAL"
        })

    # Note: Real API interaction requires handling batch limits (usually 500 items/call)
    # and valid Vector UIDs. This is a structural example.
    print(f"Prepared payload for {len(payload_items)} items.")
    print("Status: Ready to Send (Commented out for safety)")
    
    # response = requests.post(url, headers=headers, json=payload_items)
    # if response.status_code == 200:
    #    print("Upload Success! Sequences are now in your Twist account.")
    # else:
    #    print(f"Upload Failed: {response.text}")
    """
    Submits the optimized sequence to Twist Bioscience API.
    Note: This requires a valid 'Vector ID' and 'Insertion Point ID' 
    which you must fetch from your Twist account if cloning into a vector.
    """
    print(f"\n--- Preparing Submission for {gene_name} ---")
    
    # Payload structure based on Twist API documentation for Clonal Genes
    # NOTE: You normally need to fetch your specific 'vector_uid' first.
    payload = {
        "name": gene_name,
        "type": "CLONED_GENE", # or "NON_CLONED_GENE" for fragments
        "sequences": [dna_seq],
        "adapters_on": False,
        # "vector_mes_uid": "YOUR_VECTOR_ID_HERE",  <-- REQUIRED for Clonal Genes
        # "insertion_point_mes_uid": "YOUR_INSERTION_ID_HERE" <-- REQUIRED for Clonal Genes
    }

    headers = {
        "Authorization": f"Bearer {API_TOKEN}",
        "X-End-User-Token": API_TOKEN,
        "Content-Type": "application/json"
    }

    # API Endpoint for creating a construct
    url = f"{API_BASE_URL}/users/{USER_EMAIL}/constructs/"

    try:
        # Uncomment the line below to actually send the request
        # response = requests.post(url, headers=headers, json=payload)
        # response.raise_for_status()
        
        # Simulating a successful response for this demo
        print("Payload ready for submission:")
        print(json.dumps(payload, indent=2))
        print("\n(Actual submission commented out. Add your API Token to enable.)")
        
    except requests.exceptions.RequestException as e:
        print(f"Error submitting to Twist: {e}")

In [13]:
# 0. Setup Folder
setup_output_folder(OUTPUT_FOLDER)

# Generate
print("--- 1. Generating Variants ---")
variants = generate_combinations(CANONICAL_SEQ, DESIGN_SPECS, mode=GENERATION_MODE)

final_output = []

# Optimize
print(f"{'VARIANT NAME':<40} | {'STATUS'}")
print("-" * 60)

for var in variants:
    # Optimize
    opt_dna = optimize_variant(var)
    
    final_output.append({
        "name": var['name'],
        "aa_seq": var['aa_sequence'],
        "dna": opt_dna
    })
    print(f"{var['name']:<40} | Optimized")

# Validate & Save (Targeting the Output Folder)
validate_and_report(final_output, OUTPUT_FOLDER)
save_output_files(final_output, OUTPUT_FOLDER)

Using existing output directory: ../Local/Twist_Order_Dec2025
--- 1. Generating Variants ---
   > Mode: Positional (Adding variants, others stay WT)
VARIANT NAME                             | STATUS
------------------------------------------------------------
  > Optimizing codons for s_cerevisiae...
Optimizing Var_AB_LOOP-TLPDGSKE...
Var_AB_LOOP-TLPDGSKE                     | Optimized
  > Optimizing codons for s_cerevisiae...
Optimizing Var_AB_LOOP-TLPDGSKE...
Var_AB_LOOP-TLPDGSKE                     | Optimized
  > Optimizing codons for s_cerevisiae...
Optimizing Var_AB_LOOP-KNPDGTLT...
Var_AB_LOOP-KNPDGTLT                     | Optimized
  > Optimizing codons for s_cerevisiae...
Optimizing Var_AB_LOOP-KGPTGE...
Var_AB_LOOP-KGPTGE                       | Optimized
  > Optimizing codons for s_cerevisiae...
Optimizing Var_AB_LOOP-KGPYGE...
Var_AB_LOOP-KGPYGE                       | Optimized
  > Optimizing codons for s_cerevisiae...
Optimizing Var_AB_LOOP-KGKYGE...
Var_AB_LOOP-KGKYGE 